# LASSO Regression — Code Notebook
Group 11 · Jennifer Lim · Palwinder Singh · Jonathan Elahee

## Section 1 — Theory visual (Jennifer)
Compare the L1 (LASSO) and L2 (Ridge) penalties.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

beta = np.linspace(-2, 2, 400)

plt.figure(figsize=(7, 4))
plt.plot(beta, np.abs(beta),  label='L1  |β|  (LASSO)', linewidth=2)
plt.plot(beta, beta ** 2,     label='L2  β²   (Ridge)', linewidth=2, linestyle='--')
plt.axvline(0, color='gray', linewidth=0.5)
plt.title('L1 vs L2 Regularization Penalty')
plt.xlabel('Coefficient (β)'); plt.ylabel('Penalty')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Section 2 — Implementation (Palwinder)
Build LASSO from scratch with gradient descent.

In [ ]:
# Step 1 — imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)  # reproducible results

In [ ]:
# Step 2 — load dataset (8 numerical features, target = median house value)
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='MedHouseVal')
X.head()

In [ ]:
# 80/20 train/test split, then standardize so the L1 penalty is fair across features
X_train, X_test, y_train, y_test = train_test_split(
    X.values, y.values, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# Step 3 — LASSO from scratch
class LassoRegression:
    def __init__(self, learning_rate=0.01, lambda_param=0.1, n_iterations=1000):
        self.learning_rate = learning_rate
        self.lambda_param  = lambda_param   # λ — strength of the L1 penalty
        self.n_iterations  = n_iterations
        self.weights = None
        self.bias    = None
        self.loss_history = []

    def _loss(self, X, y):
        m = X.shape[0]
        y_pred = X @ self.weights + self.bias
        mse = (1 / (2 * m)) * np.sum((y_pred - y) ** 2)
        l1  = self.lambda_param * np.sum(np.abs(self.weights))
        return mse + l1

    def fit(self, X, y):
        m, n = X.shape
        self.weights = np.zeros(n)   # start coefficients at zero
        self.bias    = 0.0
        self.loss_history = []

        for _ in range(self.n_iterations):
            error = (X @ self.weights + self.bias) - y

            # MSE gradient + L1 sub-gradient (sign of weights) — this is the LASSO part
            dw = (1 / m) * (X.T @ error) + self.lambda_param * np.sign(self.weights)
            db = (1 / m) * np.sum(error)

            self.weights -= self.learning_rate * dw
            self.bias    -= self.learning_rate * db
            self.loss_history.append(self._loss(X, y))
        return self

    def predict(self, X):
        return X @ self.weights + self.bias

In [ ]:
# Step 4 — train
model = LassoRegression(learning_rate=0.05, lambda_param=0.1, n_iterations=1500)
model.fit(X_train_scaled, y_train)

print(f'Final training loss: {model.loss_history[-1]:.4f}')
print(f'Bias (intercept):    {model.bias:.4f}')

# Loss curve — should drop quickly then flatten = converged
plt.figure(figsize=(7, 4))
plt.plot(model.loss_history, color='#2E86AB', linewidth=2)
plt.title('Training Loss Convergence')
plt.xlabel('Iteration'); plt.ylabel('Cost J(β)')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Section 3 — Evaluation (Jonathan)
Test the model, inspect coefficients, visualize results.

In [ ]:
# Predictions on held-out data
y_pred_test  = model.predict(X_test_scaled)
y_pred_train = model.predict(X_train_scaled)

metrics = {
    'R² (test)':  r2_score(y_test,  y_pred_test),
    'R² (train)': r2_score(y_train, y_pred_train),  # close to test R² → not overfitting
    'MSE (test)': mean_squared_error(y_test, y_pred_test),
    'MAE (test)': mean_absolute_error(y_test, y_pred_test),
}
for name, value in metrics.items():
    print(f'{name:<12s}: {value:.4f}')

In [ ]:
# Coefficients ranked by magnitude — features with values near 0 were effectively dropped
coef_df = pd.DataFrame({
    'Feature': data.feature_names,
    'Coefficient': model.weights,
    '|Coefficient|': np.abs(model.weights),
}).sort_values('|Coefficient|', ascending=False)
print(coef_df.to_string(index=False))

colors = ['#2E86AB' if c >= 0 else '#E63946' for c in coef_df['Coefficient']]
plt.figure(figsize=(8, 4))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.6)
plt.title('LASSO Feature Coefficients')
plt.xlabel('Coefficient value')
plt.gca().invert_yaxis(); plt.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

In [ ]:
# Predicted vs actual — points should hug the diagonal
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_test, alpha=0.3, s=15, color='#2E86AB', label='Test predictions')
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
plt.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title('Predicted vs Actual')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Regularization path — sweep λ to show coefficients shrinking as the penalty grows
lambdas = [0.001, 0.01, 0.05, 0.1, 0.3, 0.6, 1.0]
paths = []
for lam in lambdas:
    m = LassoRegression(learning_rate=0.05, lambda_param=lam, n_iterations=1500)
    m.fit(X_train_scaled, y_train)
    paths.append(m.weights.copy())
paths = np.array(paths)

plt.figure(figsize=(8, 5))
for i, name in enumerate(data.feature_names):
    plt.plot(lambdas, paths[:, i], marker='o', label=name)
plt.xscale('log')
plt.axhline(0, color='black', linewidth=0.6)
plt.xlabel('λ (log scale)'); plt.ylabel('Coefficient')
plt.title('LASSO Regularization Path')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()